## 프롬프트 엔지니어링 (Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 ‘인컨텍스트 러닝(In-context Learning)’ 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

### 프롬프트 핵심 구성 요소

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

- 지시문 (Instruction): 모델이 수행해야 할 구체적인 작업 (예: 요약하라, 분류하라, 번역하라 등)
- 문맥 (Context): 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건
- 입력 데이터 (Input Data): 처리가 필요한 실제 데이터
- 출력 지시자 (Output Indicator): 결과물의 형식이나 스타일 지정 (예: 표로 정리하라, JSON 포맷으로 출력하라 등)

### 프롬프트 엔지니어링의 중요성

- 성능 최적화: 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
- 비용 효율성: 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
- 한계 극복: 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합)하더라도 유도할 수 있다.

## 환경설정
- OpenAI에서 api key 발급 받아 .env 파일에 저장 (api key 노출 되지 않도록 유의)
- 환경 변수 로드 가능한 'python-dotenv'와 api 호출을 위한 openai 설치

In [2]:
# %pip install openai python-dotenv

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# 다수의 api를 모두 호출할 수 있는 client 객체
client = OpenAI()

In [4]:
response = client.chat.completions.create(
  model="gpt-4.1-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "Korean News Article Headline Corrector\n\nYour task is to enhance the clarity, accuracy, grammar, and style of Korean news article headlines, ensuring adherence to professional news writing conventions. Do not alter the meaning of the headline. Revise to improve precision and professionalism as needed.\n\nSteps:\n- Review the given headline for ambiguous wording, grammatical errors, spelling, punctuation, spacing, readability, and adherence to journalistic standards.\n- Make meaningful corrections where possible (spacing, spelling, punctuation, style).\n- If the headline is already correct and optimal, return it unchanged.\n- Provide only the corrected headline with no explanations or extra commentary.\n\n# Output Format\n\nReturn only the revised headline in plain text. If no correction is needed, return the original headline unchanged. Do not use code blocks or provide explanations.\n\n# Examples\n\nInput: 원본 제목: \"정부, 내년 예산안 발표\"\nOutput: 정부, 내년 예산안 발표\n\nInput: 원본 제목: \"신종 코로나 바이러스 확사세, 신규 확진자 500명 돌파\"\nOutput: 신종 코로나바이러스 확산세, 신규 확진자 500명 돌파\n\nInput: 원본 제목: \"대통령, 회담 위해 중국 방문 할 예정\"\nOutput: 대통령, 회담 위해 중국 방문할 예정\n\n# Notes\n\n- Your primary goal is to make concise, professional corrections to Korean news headlines, returning only the corrected version with no extra explanation.\n- If no correction is required, return the original headline as is.\n- Never explain your corrections unless explicitly asked."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "input: \n어제 서울에서의 큰불이나서 수백만명이 대피했다."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "어제 서울에서 큰불 나 수백만 명 대피했다"
        }
      ]
    }
  ],
  response_format={
    "type": "text"
  },
  temperature=1,
  max_completion_tokens=2048,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  store=False
)

In [5]:
response

ChatCompletion(id='chatcmpl-DcPMfmx2Bc1qFtHRqMQHo8doIqZjT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='어제 서울에서 큰불 나 수백만 명 대피했다', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1778046337, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_44222b3f42', usage=CompletionUsage(completion_tokens=14, prompt_tokens=370, total_tokens=384, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [6]:
response.choices[0].message.content

'어제 서울에서 큰불 나 수백만 명 대피했다'

In [7]:
# inputdata 수정 테스트
response = client.chat.completions.create(
  model="gpt-4.1-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "Korean News Article Headline Corrector\n\nYour task is to enhance the clarity, accuracy, grammar, and style of Korean news article headlines, ensuring adherence to professional news writing conventions. Do not alter the meaning of the headline. Revise to improve precision and professionalism as needed.\n\nSteps:\n- Review the given headline for ambiguous wording, grammatical errors, spelling, punctuation, spacing, readability, and adherence to journalistic standards.\n- Make meaningful corrections where possible (spacing, spelling, punctuation, style).\n- If the headline is already correct and optimal, return it unchanged.\n- Provide only the corrected headline with no explanations or extra commentary.\n\n# Output Format\n\nReturn only the revised headline in plain text. If no correction is needed, return the original headline unchanged. Do not use code blocks or provide explanations.\n\n# Examples\n\nInput: 원본 제목: \"정부, 내년 예산안 발표\"\nOutput: 정부, 내년 예산안 발표\n\nInput: 원본 제목: \"신종 코로나 바이러스 확사세, 신규 확진자 500명 돌파\"\nOutput: 신종 코로나바이러스 확산세, 신규 확진자 500명 돌파\n\nInput: 원본 제목: \"대통령, 회담 위해 중국 방문 할 예정\"\nOutput: 대통령, 회담 위해 중국 방문할 예정\n\n# Notes\n\n- Your primary goal is to make concise, professional corrections to Korean news headlines, returning only the corrected version with no extra explanation.\n- If no correction is required, return the original headline as is.\n- Never explain your corrections unless explicitly asked."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "input: \n두쫀쿠 열풍 뭐가 존맛탱리하는지 알 수가 없네"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "어제 서울에서 큰불 나 수백만 명 대피했다"
        }
      ]
    }
  ],
  response_format={
    "type": "text"
  },
  temperature=1,
  max_completion_tokens=2048,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  store=False
)

In [8]:
response.choices[0].message.content

'두쫀쿠 열풍, 무엇이 정말 맛있는지 알기 어렵네'

In [9]:
# 재사용을 위한 함수 선언
def correct_headline(headline,model="gpt-4.1-mini",temperature=1,max_completion_tokens=2048,top_p=1):
  system_message = "Korean News Article Headline Corrector\n\nYour task is to enhance the clarity, accuracy, grammar, and style of Korean news article headlines, ensuring adherence to professional news writing conventions. Do not alter the meaning of the headline. Revise to improve precision and professionalism as needed.\n\nSteps:\n- Review the given headline for ambiguous wording, grammatical errors, spelling, punctuation, spacing, readability, and adherence to journalistic standards.\n- Make meaningful corrections where possible (spacing, spelling, punctuation, style).\n- If the headline is already correct and optimal, return it unchanged.\n- Provide only the corrected headline with no explanations or extra commentary.\n\n# Output Format\n\nReturn only the revised headline in plain text. If no correction is needed, return the original headline unchanged. Do not use code blocks or provide explanations.\n\n# Examples\n\nInput: 원본 제목: \"정부, 내년 예산안 발표\"\nOutput: 정부, 내년 예산안 발표\n\nInput: 원본 제목: \"신종 코로나 바이러스 확사세, 신규 확진자 500명 돌파\"\nOutput: 신종 코로나바이러스 확산세, 신규 확진자 500명 돌파\n\nInput: 원본 제목: \"대통령, 회담 위해 중국 방문 할 예정\"\nOutput: 대통령, 회담 위해 중국 방문할 예정\n\n# Notes\n\n- Your primary goal is to make concise, professional corrections to Korean news headlines, returning only the corrected version with no extra explanation.\n- If no correction is required, return the original headline as is.\n- Never explain your corrections unless explicitly asked."
  model = model
  user_message = f"입력 제목 : {headline}"
  response = client.chat.completions.create(
  model=model,
  messages=[
      {
        "role": "system",
        "content": [
          {
            "type": "text",
            "text": system_message
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": user_message
          }
        ]
      },
      {
        "role": "assistant",
        "content": [
          {
            "type": "text",
            "text": "어제 서울에서 큰불 나 수백만 명 대피했다"
          }
        ]
      }
    ],
    response_format={
      "type": "text"
    },
    temperature=temperature,
    max_completion_tokens=max_completion_tokens,
    top_p=top_p ,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
  )
  return response.choices[0].message.content

In [10]:
correct_headline('두쫀쿠 뭐가 존맛탱이라는지 알 수가 없네',model="gpt-5.4")

'두찜, 뭐가 ‘존맛탱’이라는지 알 수가 없네'

In [11]:
# 여러 건의 제목 수정
headlines = [
    '주말 성수동 거리에는 커플들이 그득그득 하더라',
    '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요',
    '빡센 직업으로 끼니 거르기 일쑤인 노동자들의 애환'
]

for headline in headlines:
    print(correct_headline(headline))
    print()

주말 성수동 거리에는 커플들로 가득했다

유튜버의 생존일기, 쉽지 않은 그들의 여정에 함께해요

빡센 직업으로 끼니 거르기 일쑤인 노동자들의 애환



## JSON Object 반환으로 변경
기자들이 송고한 제목에서 맞춤법/문법/의미/어조 등을 고려해 최상의 뉴스 제목을 뽑아내는 20년 경력의 뉴스데스크장입니다.

## Instruction
교정이 필요한 기사 제목을 입력받아 맞춤법, 띄어쓰기, 문법, 의미, 어조를 점검하고 더 나은 뉴스 제목으로 교정하세요.

아래 기준에 따라 작업합니다.

1. 입력된 기사 제목을 분석하여 맞춤법 오류, 띄어쓰기 오류, 문법 오류, 의미상 어색한 표현, 과도하게 감정적이거나 부정적인 표현을 찾습니다.
2. 오류가 있다면 모두 반영하여 간결하고 명확한 뉴스 제목으로 수정합니다.
3. 비속어/욕설이 포함되어 있다면 제거하고, 의미가 전달되는 중립적 표현으로 수정합니다.
4. 오류가 여러 개 있을 경우 각각의 교정 이유를 구분하여 작성합니다.
5. 오류가 없더라도 더 자연스럽고 기사 제목에 적합한 표현이 있다면 다듬을 수 있습니다.
6. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
7. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## Output JSON Format
{
 "original_title": "입력된 원제목",
 "corrected_title": "교정된 제목",
 "correction_reasons":
  [
    {"original": "교정 전 표현",
     "corrected": "교정 후 표현",
     "reason": "교정 이유"}
  ] 
}

## Rules
- original_title에는 사용자가 입력한 제목을 그대로 작성합니다.
- corrected_title에는 최종 교정 제목만 작성합니다.
- correction_reasons는 배열로 작성합니다.
- 교정할 부분이 없다면 correction_reasons는 빈 배열([])로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.

In [ ]:
import json

# 재사용을 위한 함수 선언
def correct_headline_json(headline,model="gpt-4.1-mini",temperature=1,max_completion_tokens=2048,top_p=1):
  system_message = """
  기자들이 송고한 제목에서 맞춤법/문법/의미/어조 등을 고려해 최상의 뉴스 제목을 뽑아내는 20년 경력의 뉴스데스크장입니다.

## Instruction
교정이 필요한 기사 제목을 입력받아 맞춤법, 띄어쓰기, 문법, 의미, 어조를 점검하고 더 나은 뉴스 제목으로 교정하세요.

아래 기준에 따라 작업합니다.

1. 입력된 기사 제목을 분석하여 맞춤법 오류, 띄어쓰기 오류, 문법 오류, 의미상 어색한 표현, 과도하게 감정적이거나 부정적인 표현을 찾습니다.
2. 오류가 있다면 모두 반영하여 간결하고 명확한 뉴스 제목으로 수정합니다.
3. 비속어/욕설이 포함되어 있다면 제거하고, 의미가 전달되는 중립적 표현으로 수정합니다.
4. 오류가 여러 개 있을 경우 각각의 교정 이유를 구분하여 작성합니다.
5. 오류가 없더라도 더 자연스럽고 기사 제목에 적합한 표현이 있다면 다듬을 수 있습니다.
6. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
7. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## Output JSON Format
{
 "original_title": "입력된 원제목",
 "corrected_title": "교정된 제목",
 "correction_reasons":
  [
    {"original": "교정 전 표현",
     "corrected": "교정 후 표현",
     "reason": "교정 이유"}
  ] 
}

## Rules
- original_title에는 사용자가 입력한 제목을 그대로 작성합니다.
- corrected_title에는 최종 교정 제목만 작성합니다.
- correction_reasons는 배열로 작성합니다.
- 교정할 부분이 없다면 correction_reasons는 빈 배열([])로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.
"""
  model = model
  user_message = f"입력 제목 : {headline}"
  response = client.chat.completions.create(
  model=model,
  messages=[
      {
        "role": "system",
        "content": [
          {
            "type": "text",
            "text": system_message
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": user_message
          }
        ]
      },
      {
        "role": "assistant",
        "content": [
          {
            "type": "text",
            "text": "어제 서울에서 큰불 나 수백만 명 대피했다"
          }
        ]
      }
    ],
    response_format={
      "type": "json_object"
    },
    temperature=temperature,
    max_completion_tokens=max_completion_tokens,
    top_p=top_p ,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
  )
  return json.loads(response.choices[0].message.content)

In [13]:
# 여러 건의 제목 수정
headlines = [
    '주말 성수동 거리에는 커플들이 그득그득 하더라',
    '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요',
    '빡센 직업으로 끼니 거르기 일쑤인 노동자들의 애환'
]

for headline in headlines:
    print(correct_headline_json(headline))
    print()

{'original_title': '주말 성수동 거리에는 커플들이 그득그득 하더라', 'corrected_title': '주말 성수동 거리에는 커플들로 북적였다', 'correction_reasons': [{'original': '그득그득 하더라', 'corrected': '커플들로 북적였다', 'reason': "'그득그득 하더라'는 비격식적이고 어색한 표현이므로, 공식 뉴스 제목에는 더 명확하고 간결한 '커플들로 북적였다'로 수정함"}]}

{'original_title': '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요', 'corrected_title': '유튜버의 생존 일기, 쉽지 않은 그들의 여정에 함께합니다', 'correction_reasons': [{'original': '유투버', 'corrected': '유튜버', 'reason': "표준어인 '유튜버'로 교정"}, {'original': '생존일기', 'corrected': '생존 일기', 'reason': '합성어의 올바른 띄어쓰기 적용'}, {'original': '함께해요', 'corrected': '함께합니다', 'reason': '공식 뉴스 제목에 어울리는 격식 있는 표현으로 수정'}]}

{'original_title': '빡센 직업으로 끼니 거르기 일쑤인 노동자들의 애환', 'corrected_title': '고된 노동으로 끼니 거르기 일쑤인 노동자들의 애환', 'correction_reasons': [{'original': '빡센 직업', 'corrected': '고된 노동', 'reason': "비속어인 '빡센'을 공식적이고 중립적인 표현인 '고된'으로 교체함"}]}



## 냉털 마스터
- 사용자는 냉장고에 남아있는 음식 재료를 input data로 전달한다.
- 해당 음식 재료를 기반으로 어떤 음식을 만들지 조언(레시피 포함)한다.
- 적절한 프롬프팅을 통해 구현한다.
- 응답 양식은 json 형태로 구현한다.

In [16]:
def fridge_raid_master(user_foods):
  response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
      {
        "role": "system",
        "content": [
          {
            "type": "text",
            "text": "너는 전세계의 모든 레시피를 알고 있는 5성급 레스토랑의 쉐프이다.\n\n## Instruction\n냉장고에 남아있는 음식 재료를 input data로 주면 해당 재료들을 기반으로 어떤 음식으로 만들지 조언(레시피 포함)해 준다.\n\n1. input  재료가 들어간 음식이여야 한다.\n2. 어떤 재료가 얼만큼(그람, 큰술, 작은술,  컵 등) 들어가는지 명확히 알려준다.\n3. 레시피를 무조건 제시한다.\n4. 레시피는 순서대로 시간이나 그람 수를 상세히 알려준다. \n5 응답은 반드시 유효한 JSON 객체 하나만 반환합니다. \n6. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.\n\n## Output JSON Format\r\n{\r\n \"input_data\": \"입력 재료\", \n \"food_name\": \"음식 이름\", \n \"recipe\": 레시피\n  [\r\n    {\"필요한 재료\": \"필요한 재료 내용\",\n     \"1\": \"레시피 1번 내용\", \n     \"2\": \"레시피 2번 내용\", \n     \"3\": \"레시피 3번 내용\"} \n  ] \r\n}\n\n## Rules\r\n- input_data에는 사용자가 입력한 재료를 그대로 작성합니다. \n- food_name에는 음식 이름만 작성합니다. \n- recipe는 배열로 작성합니다. \n- 레시피의 마지막 내용의 시작은 \"마지막으로\"를 사용한다\n- 모든 문자열은 한국어로 작성합니다."
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "input_data: ['소고기','양파','표고버섯','명란젓','된장']"
          }
        ]
      },
      {
        "role": "assistant",
        "content": [
          {
            "type": "text",
            "text": "{\n  \"input_data\": [\n    \"소고기\",\n    \"양파\",\n    \"표고버섯\",\n    \"명란젓\",\n    \"된장\"\n  ],\n  \"food_name\": \"소고기 된장찌개 명란젓 토핑\",\n  \"recipe\": [\n    {\n      \"1\": \"소고기 200g을 한입 크기로 썰고, 양파 1개는 채 썰어 준비합니다.\",\n      \"2\": \"표고버섯 3~4개는 기둥을 제거하고 얇게 썰어줍니다.\",\n      \"3\": \"냄비에 물 700ml를 붓고 끓으면 된장 3큰술을 풀어 국물 베이스를 만듭니다.\",\n      \"4\": \"된장 국물이 끓으면 준비한 소고기, 양파, 표고버섯을 넣고 중불에서 10분간 끓입니다.\",\n      \"5\": \"명란젓 2큰술은 껍질을 제거하고 고기와 야채 위에 올려 살짝 익힙니다.\",\n      \"6\": \"마지막으로 간을 보고 필요시 소금이나 국간장으로 간을 맞추고, 한소끔 더 끓여서 완성합니다.\"\n    }\n  ]\n}"
          }
        ]
      }
    ],
    response_format={
      "type": "json_object"
    },
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
  )
  return json.loads(response.choices[0].message.content)

In [17]:
user_foods = ['소고기','양파','표고버섯','명란젓','된장']
fridge_raid_master(user_foods)

{'input_data': ['소고기', '양파', '표고버섯', '명란젓', '된장'],
 'food_name': '명란 된장 소고기찌개',
 'recipe': [{'필요한 재료': '소고기 200g, 양파 1개, 표고버섯 4개, 명란젓 2큰술, 된장 3큰술, 물 800ml',
   '1': '소고기 200g은 한입 크기로 썬다.',
   '2': '양파 1개는 채썰고, 표고버섯 4개는 얇게 썬다.',
   '3': '냄비에 물 800ml를 붓고 끓으면 된장 3큰술을 잘 풀어 넣는다.',
   '4': '된장물이 끓으면 소고기, 양파, 표고버섯을 넣고 중불에서 10분간 끓인다.',
   '5': '명란젓 2큰술은 껍질을 제거하고 찌개 위에 올려 2분간 익힌다.',
   '6': '마지막으로 간을 보고 필요하면 소금 약간을 추가해 간을 맞춘 후 2분 더 끓여 완성한다.'}]}

## [참고] JSON Schema 기반 응답 형식 지정

`response_format={"type": "json_schema"}`는 모델의 응답을 정해진 JSON 구조에 맞게 받기 위해 사용한다.

기존의 `json_object`는 “JSON 객체로 응답하라”는 정도만 보장한다.

    response_format={
        "type": "json_object"
    }

하지만 이 방식은 다음을 엄격하게 보장하지는 못한다.

- 필요한 필드가 모두 있는지
- 필드명이 정확한지
- 값의 자료형이 맞는지
- 배열 안의 값 형태가 맞는지
- 정해진 값만 사용했는지

그래서 응답 결과를 코드에서 바로 사용해야 할 때는 `json_schema`를 사용하는 것이 더 안정적이다.

## 기본 구조

    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "fridge_recipe_schema",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "recommended_dish": {
                        "type": "string"
                    }
                },
                "required": ["recommended_dish"],
                "additionalProperties": False
            }
        }
    }

## 주요 작성법

### 1. name

스키마의 이름을 지정한다.

    "name": "fridge_recipe_schema"

### 2. strict

스키마를 엄격하게 따르도록 설정한다.

    "strict": True

### 3. type

값의 자료형을 지정한다.

    "type": "object"

자주 사용하는 타입은 다음과 같다.

    object  : JSON 객체
    array   : 배열
    string  : 문자열
    number  : 숫자
    integer : 정수
    boolean : 참/거짓

### 4. properties

JSON 객체 안에 들어갈 필드를 정의한다.

    "properties": {
        "recommended_dish": {
            "type": "string"
        },
        "dish_summary": {
            "type": "string"
        }
    }

### 5. required

반드시 포함되어야 하는 필드를 지정한다.

    "required": [
        "recommended_dish",
        "dish_summary"
    ]

### 6. array와 items

배열은 `array`, 배열 안의 값은 `items`로 지정한다.

    "input_ingredients": {
        "type": "array",
        "items": {
            "type": "string"
        }
    }

위 설정은 다음과 같은 값을 기대한다.

    "input_ingredients": ["소고기", "양파", "표고버섯"]

### 7. enum

허용할 값을 제한할 때 사용한다.

    "difficulty": {
        "type": "string",
        "enum": ["쉬움", "보통", "어려움"]
    }

위 설정을 사용하면 `difficulty`에는 `"쉬움"`, `"보통"`, `"어려움"` 중 하나만 들어갈 수 있다.

### 8. minItems

배열의 최소 개수를 지정한다.

    "steps": {
        "type": "array",
        "items": {
            "type": "string"
        },
        "minItems": 4
    }

위 설정은 `steps` 배열에 최소 4개의 값이 필요하다는 뜻이다.

### 9. additionalProperties

정의하지 않은 필드를 허용할지 결정한다.

    "additionalProperties": False

`False`로 설정하면 스키마에 없는 필드가 추가되는 것을 막을 수 있다.

## 현재 예제에서의 역할

냉장고 레시피 예제에서는 다음 구조를 일정하게 유지하기 위해 `json_schema`를 사용한다.

    {
        "input_ingredients": [],
        "recommended_dish": "",
        "dish_summary": "",
        "used_ingredients": [],
        "additional_ingredients": [],
        "unused_ingredients": [],
        "recipe": {
            "servings": "",
            "estimated_time": "",
            "difficulty": "",
            "steps": []
        },
        "taste_profile": "",
        "why_recommended": "",
        "tips": []
    }

이렇게 구조를 고정해두면 응답을 받은 뒤 바로 Python 객체로 변환해서 사용하기 쉽다.

In [18]:
import json

def fridge_raid_master(
    user_foods,
    model="gpt-4.1-mini",
    temperature=0.7,
    max_completion_tokens=2048,
    top_p=1
):
    system_message = """
당신은 냉장고 속 남은 재료를 바탕으로 현실적인 한 끼를 추천하는 요리 도우미입니다.

사용자는 냉장고에 남아 있는 재료 목록을 리스트 형태로 전달합니다.
당신은 입력 재료를 분석하여 실제로 만들기 좋은 요리 1개와 간단한 레시피를 JSON 형식으로 반환해야 합니다.

## 판단 기준

1. 입력 재료 중 자연스럽게 어울리는 조합을 우선 선택합니다.
2. 모든 재료를 억지로 사용하지 않습니다.
3. 향이나 염도가 강한 재료는 다른 재료와 충돌할 경우 제외할 수 있습니다.
   예: 명란젓, 된장, 김치, 고추장, 젓갈류 등
4. 주재료와 부재료를 구분하여 요리를 설계합니다.
5. 초보자도 만들 수 있는 가정식 수준의 요리만 추천합니다.
6. 튀김, 오븐 요리, 장시간 숙성, 전문 장비가 필요한 조리법은 피합니다.
7. 추가 재료는 일반 가정에 있을 가능성이 높은 기본 양념으로 제한합니다.
   예: 소금, 후추, 식용유, 물, 간장, 설탕, 다진 마늘, 고춧가루, 참기름
8. 입력 재료 중 사용하지 않는 재료가 있다면 unused_ingredients에 이유를 작성합니다.
9. 추천 요리는 1개만 제안합니다.
10. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
11. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## 추천 우선순위

1순위: 입력 재료만으로 자연스럽게 만들 수 있는 요리
2순위: 기본 양념만 추가하면 만들 수 있는 요리
3순위: 일부 재료를 제외하더라도 맛의 완성도가 높은 요리

## Output JSON Format

{
  "input_ingredients": ["사용자가 입력한 재료"],
  "recommended_dish": "추천 요리명",
  "dish_summary": "요리에 대한 한 줄 설명",
  "used_ingredients": ["실제로 사용하는 입력 재료"],
  "additional_ingredients": ["추가로 필요한 기본 재료"],
  "unused_ingredients": [
    {
      "ingredient": "사용하지 않은 재료",
      "reason": "사용하지 않은 이유"
    }
  ],
  "recipe": {
    "servings": "몇 인분인지",
    "estimated_time": "예상 조리 시간",
    "difficulty": "쉬움 | 보통 | 어려움",
    "steps": [
      "1단계 조리 설명",
      "2단계 조리 설명",
      "3단계 조리 설명",
      "4단계 조리 설명"
    ]
  },
  "taste_profile": "맛의 특징",
  "why_recommended": "이 요리를 추천하는 이유",
  "tips": [
    "조리 팁 1",
    "조리 팁 2"
  ]
}

## 출력 규칙

- input_ingredients에는 사용자가 입력한 재료를 그대로 작성합니다.
- used_ingredients에는 입력 재료 중 실제로 사용하는 재료만 작성합니다.
- additional_ingredients에는 입력에 없지만 필요한 기본 재료만 작성합니다.
- 추가 재료가 필요 없다면 additional_ingredients는 빈 배열([])로 작성합니다.
- 사용하지 않는 입력 재료가 없다면 unused_ingredients는 빈 배열([])로 작성합니다.
- recipe.steps는 최소 4단계 이상 작성합니다.
- difficulty는 반드시 "쉬움", "보통", "어려움" 중 하나로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.
"""

    user_message = f"""
다음은 사용자가 냉장고에 가지고 있는 재료입니다.

{user_foods}

위 재료를 바탕으로 실제로 만들기 좋은 요리 1개를 추천하세요.
모든 재료를 억지로 사용하지 말고, 가장 맛이 자연스러운 조합을 선택하세요.
응답은 지정된 JSON 형식으로만 작성하세요.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": system_message
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": user_message
                    }
                ]
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "fridge_recipe_schema",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "input_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "recommended_dish": {
                            "type": "string"
                        },
                        "dish_summary": {
                            "type": "string"
                        },
                        "used_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "additional_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "unused_ingredients": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "ingredient": {"type": "string"},
                                    "reason": {"type": "string"}
                                },
                                "required": ["ingredient", "reason"],
                                "additionalProperties": False
                            }
                        },
                        "recipe": {
                            "type": "object",
                            "properties": {
                                "servings": {"type": "string"},
                                "estimated_time": {"type": "string"},
                                "difficulty": {
                                    "type": "string",
                                    "enum": ["쉬움", "보통", "어려움"]
                                },
                                "steps": {
                                    "type": "array",
                                    "items": {"type": "string"},
                                    "minItems": 4
                                }
                            },
                            "required": [
                                "servings",
                                "estimated_time",
                                "difficulty",
                                "steps"
                            ],
                            "additionalProperties": False
                        },
                        "taste_profile": {
                            "type": "string"
                        },
                        "why_recommended": {
                            "type": "string"
                        },
                        "tips": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "input_ingredients",
                        "recommended_dish",
                        "dish_summary",
                        "used_ingredients",
                        "additional_ingredients",
                        "unused_ingredients",
                        "recipe",
                        "taste_profile",
                        "why_recommended",
                        "tips"
                    ],
                    "additionalProperties": False
                }
            }
        },
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return json.loads(response.choices[0].message.content)

In [19]:
user_foods = ['소고기', '양파', '표고버섯', '명란젓', '된장']
fridge_raid_master(user_foods)

{'input_ingredients': ['소고기', '양파', '표고버섯', '명란젓', '된장'],
 'recommended_dish': '소고기 표고버섯 된장볶음',
 'dish_summary': '소고기와 표고버섯, 양파를 된장 양념으로 간단히 볶아낸 가정식 반찬',
 'used_ingredients': ['소고기', '양파', '표고버섯', '된장'],
 'additional_ingredients': ['식용유', '설탕', '다진 마늘', '후추'],
 'unused_ingredients': [{'ingredient': '명란젓',
   'reason': '된장과 명란젓의 강한 향과 염도가 겹쳐 맛이 충돌할 수 있어 사용하지 않음'}],
 'recipe': {'servings': '2인분',
  'estimated_time': '20분',
  'difficulty': '쉬움',
  'steps': ['소고기는 먹기 좋은 크기로 썰고, 양파와 표고버섯은 채 썬다.',
   '팬에 식용유를 두르고 다진 마늘을 넣어 향을 낸다.',
   '소고기를 넣고 중불에서 익을 때까지 볶는다.',
   '양파와 표고버섯을 넣고 함께 볶다가 된장과 설탕, 후추를 넣어 간을 맞춘다.',
   '재료가 잘 익고 양념이 어우러지면 불을 끄고 그릇에 담아 낸다.']},
 'taste_profile': '구수하고 짭조름하며 감칠맛 나는 된장 양념에 고기와 버섯의 풍미가 조화로운 맛',
 'why_recommended': '된장과 소고기, 표고버섯은 서로 어울리는 재료로, 명란젓을 제외해 맛의 균형을 맞춘 간단하고 맛있는 한 끼 반찬이 되기 때문입니다.',
 'tips': ['된장은 너무 많이 넣지 말고 조금씩 넣어 간을 조절하세요.', '표고버섯 대신 다른 버섯류를 사용해도 좋습니다.']}

In [20]:
# 여러 테스트 케이스 준비
test_food_lists = [
    ['계란', '대파', '밥', '김치'],
    ['닭가슴살', '양배추', '당근', '양파'],
    ['참치캔', '김치', '밥', '계란'],
    ['파스타면', '베이컨', '마늘', '양파'],
    ['떡', '어묵', '양배추', '대파'],
]

# 여러 번 테스트 실행
results = []

for idx, foods in enumerate(test_food_lists, start=1):
    print(f"\n===== 테스트 {idx} =====")
    print("입력 재료:", foods)

    result = fridge_raid_master(foods)

    results.append(result)

    print("추천 요리:", result["recommended_dish"])
    print("한 줄 설명:", result["dish_summary"])
    print("사용 재료:", result["used_ingredients"])
    print("추가 재료:", result["additional_ingredients"])
    print("사용하지 않은 재료:", result["unused_ingredients"])
    print("예상 시간:", result["recipe"]["estimated_time"])
    print("난이도:", result["recipe"]["difficulty"])
    print("추천 이유:", result["why_recommended"])


===== 테스트 1 =====
입력 재료: ['계란', '대파', '밥', '김치']
추천 요리: 김치계란볶음밥
한 줄 설명: 김치와 계란, 대파를 넣어 간단하게 만든 맛있는 볶음밥
사용 재료: ['계란', '대파', '밥', '김치']
추가 재료: ['식용유', '간장', '소금', '후추']
사용하지 않은 재료: []
예상 시간: 15분
난이도: 쉬움
추천 이유: 냉장고 재료만으로 쉽고 빠르게 만들 수 있으며 김치와 계란의 조합이 가장 자연스럽고 맛있기 때문

===== 테스트 2 =====
입력 재료: ['닭가슴살', '양배추', '당근', '양파']
추천 요리: 닭가슴살 양배추 볶음
한 줄 설명: 닭가슴살과 양배추를 중심으로 당근과 양파를 더해 만든 간단하고 건강한 볶음 요리입니다.
사용 재료: ['닭가슴살', '양배추', '당근', '양파']
추가 재료: ['간장', '소금', '후추', '식용유', '다진 마늘']
사용하지 않은 재료: []
예상 시간: 20분
난이도: 쉬움
추천 이유: 입력된 모든 재료가 자연스럽게 어울리며 초보자도 만들기 쉬운 건강한 가정식 볶음 요리이기 때문입니다.

===== 테스트 3 =====
입력 재료: ['참치캔', '김치', '밥', '계란']
추천 요리: 참치김치볶음밥
한 줄 설명: 참치와 김치를 넣어 간단하게 만드는 매콤한 볶음밥
사용 재료: ['참치캔', '김치', '밥', '계란']
추가 재료: ['식용유', '간장', '소금', '후추']
사용하지 않은 재료: []
예상 시간: 20분
난이도: 쉬움
추천 이유: 참치와 김치, 밥, 계란이 잘 어울려 간편하면서도 맛있는 한 끼가 되기 때문에 추천한다.

===== 테스트 4 =====
입력 재료: ['파스타면', '베이컨', '마늘', '양파']
추천 요리: 베이컨 마늘 파스타
한 줄 설명: 베이컨과 마늘의 풍미가 어우러진 간단하고 맛있는 파스타 요리입니다.
사용 재료: ['파스타면', '베이컨', '마늘', '양파']
추가 재료: ['식용유', '소금', 